# Этап 2. Интерференция двух волн

Конспект: [[PSI-02-Two-Wave-Interference]]

Цель — понять, откуда берётся формула (1) статьи

$$I = a + b\cos\varphi$$

и что физически стоит за буквами $a$, $b$, $\varphi$.

План:

0. минимум про свет как волну (то, что нужно из этапа 1);
1. сложение двух волн — анимация;
2. вывод формулы интерференции и её численная проверка;
3. фон, амплитуда и модуляция через яркости двух пучков;
4. откуда берутся **полосы**;
5. почему модуляция бывает меньше единицы.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (8, 3.5)
plt.rcParams["axes.grid"] = True
rng = np.random.default_rng(0)

lam = 633e-9  # длина волны красного He-Ne лазера, м

## 0. Минимум про свет как волну

Световая волна — это колебание электрического поля, бегущее в пространстве:

$$E(z, t) = E_0\cos\!\left(\frac{2\pi}{\lambda}z - \omega t + \varphi_0\right)$$

Три факта, которые нужны дальше.

**1. Путь превращается в фазу.** Пройдя расстояние $\Delta$, волна набирает фазу

$$\varphi = \frac{2\pi}{\lambda}\,\Delta$$

Одна длина волны пути = $2\pi$ фазы = полный оборот вектора из этапа 0. Для $\lambda = 633$ нм сдвиг на 90° — это всего 158 нм пути.

**2. Камера не видит колебаний.** Частота света около $5\cdot10^{14}$ Гц, а время экспозиции — миллисекунды. За это время поле успевает качнуться сотни миллиардов раз. Камера измеряет **интенсивность** — среднюю по времени мощность, пропорциональную квадрату амплитуды:

$$I \propto \langle E^2\rangle = \frac{E_0^2}{2}$$

Фаза одиночной волны из этой величины исчезает полностью. Проверим.

In [ ]:
t = np.linspace(0, 200, 200_001)  # 200 периодов, время в единицах периода
E0 = 1.5

for phi_deg in (0, 37, 90, 211):
    E = E0 * np.cos(2 * np.pi * t + np.deg2rad(phi_deg))
    print(f"φ = {phi_deg:3d}°   <E²> = {np.mean(E**2):.4f}   (E0²/2 = {E0**2 / 2:.4f})")

Какой бы ни была фаза, средняя мощность одна и та же. Одиночная волна фазу «не показывает».

**3. Комплексная амплитуда.** Чтобы не таскать за собой $\omega t$, волну описывают одним комплексным числом — тем самым вектором из этапа 0, остановленным в какой-то момент:

$$U = A\,e^{i\varphi}, \qquad I = |U|^2$$

Длина вектора — амплитуда, угол — фаза. Множитель $1/2$ принято прятать в определение $A$, так что интенсивность — просто квадрат модуля.

## 1. Сложение двух волн

Поля складываются **до** того, как камера усредняет мощность. Поэтому результат зависит от разности фаз двух волн:

- волны в фазе (разность 0°) — гребень на гребень, амплитуды складываются;
- волны в противофазе (180°) — гребень на впадину, амплитуды вычитаются.

In [ ]:
A1, A2 = 1.0, 0.6
x = np.linspace(0, 3, 600)                 # координата в длинах волн
dphis = np.linspace(0, 2 * np.pi, 72, endpoint=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})
ax1.plot(x, A1 * np.cos(2 * np.pi * x), "C0", lw=1, label="волна 1")
w2, = ax1.plot([], [], "C1", lw=1, label="волна 2")
ws, = ax1.plot([], [], "k", lw=2.5, label="сумма")
ax1.set_ylim(-1.8, 1.8)
ax1.set_xlabel("координата, длины волн")
ax1.set_ylabel("поле E")
ax1.legend(loc="upper right", ncol=3)
title = ax1.set_title("")

full = np.linspace(0, 2 * np.pi, 200)
ax2.plot(np.rad2deg(full), A1**2 + A2**2 + 2 * A1 * A2 * np.cos(full), "0.6")
dot, = ax2.plot([], [], "ro", ms=8)
ax2.set_xlabel("разность фаз, градусы")
ax2.set_ylabel("интенсивность суммы")
ax2.set_ylim(0, (A1 + A2) ** 2 * 1.1)


def update(i):
    dphi = dphis[i]
    e2 = A2 * np.cos(2 * np.pi * x + dphi)
    w2.set_data(x, e2)
    ws.set_data(x, A1 * np.cos(2 * np.pi * x) + e2)
    dot.set_data([np.rad2deg(dphi)], [A1**2 + A2**2 + 2 * A1 * A2 * np.cos(dphi)])
    title.set_text(f"разность фаз = {np.rad2deg(dphi):.0f}°")
    return w2, ws, dot, title


anim = FuncAnimation(fig, update, frames=len(dphis), interval=80, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

Чёрная кривая — всё та же косинусоида той же частоты, меняется только её размах. Камера увидит квадрат этого размаха — красная точка справа.

## 2. Формула интерференции

Две волны в одной точке: $U_1 = A_1e^{i\varphi_1}$, $U_2 = A_2e^{i\varphi_2}$. Интенсивность суммы:

$$I = |U_1 + U_2|^2 = |U_1|^2 + |U_2|^2 + U_1U_2^* + U_1^*U_2$$

Первые два слагаемых — интенсивности пучков по отдельности, $I_1 = A_1^2$ и $I_2 = A_2^2$. Два последних — комплексно сопряжённая пара, их сумма равна удвоенной вещественной части:

$$U_1U_2^* + U_1^*U_2 = 2A_1A_2\cos(\varphi_1-\varphi_2)$$

Итого, с $\varphi = \varphi_1 - \varphi_2$:

$$\boxed{I = I_1 + I_2 + 2\sqrt{I_1I_2}\,\cos\varphi}$$

Третье слагаемое называют **интерференционным членом**. Без него было бы просто $I_1 + I_2$ — как от двух фонариков.

Проверим формулу «в лоб»: сложим два колеблющихся поля и усредним квадрат по времени, как это делает камера.

In [ ]:
I1, I2 = 1.0, 0.36
A1, A2 = np.sqrt(2 * I1), np.sqrt(2 * I2)   # вещественные амплитуды: I = A²/2

dphi = np.linspace(0, 2 * np.pi, 25)
I_camera = []
for d in dphi:
    E = A1 * np.cos(2 * np.pi * t) + A2 * np.cos(2 * np.pi * t + d)
    I_camera.append(np.mean(E**2))

I_formula = I1 + I2 + 2 * np.sqrt(I1 * I2) * np.cos(full)

fig, ax = plt.subplots()
ax.plot(np.rad2deg(full), I_formula, "r", label=r"$I_1+I_2+2\sqrt{I_1I_2}\cos\varphi$")
ax.plot(np.rad2deg(dphi), I_camera, "ko", label="усреднение по времени")
ax.axhline(I1 + I2, color="0.5", ls="--", label=r"$I_1+I_2$ (без интерференции)")
ax.set_xlabel("разность фаз φ, градусы")
ax.set_ylabel("I")
ax.legend()
plt.show()

## 3. Фон, амплитуда и модуляция

Сравним с формулой (1) статьи $I = a + b\cos\varphi$:

| В статье | Через яркости пучков | Смысл |
|---|---|---|
| $a$ | $I_1 + I_2$ | фон — сумма яркостей двух пучков |
| $b$ | $2\sqrt{I_1I_2}$ | амплитуда колебаний яркости |
| $\varphi$ | $\varphi_1 - \varphi_2$ | **разность** фаз двух волн |

Главное про $\varphi$: интерферометр измеряет не фазу волны, а разность фаз двух волн. Одна из них — опорная, с известной формой фронта, вторая — исследуемая.

Модуляция из этапа 0:

$$\gamma = \frac{b}{a} = \frac{2\sqrt{I_1I_2}}{I_1+I_2}$$

Она зависит только от **отношения** яркостей пучков.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

for ratio in (1, 0.25, 0.04, 0.0):
    i1 = 1 / (1 + ratio)
    i2 = ratio / (1 + ratio)            # нормировка: I1 + I2 = 1
    gamma = 2 * np.sqrt(i1 * i2)
    ax1.plot(np.rad2deg(full), 1 + gamma * np.cos(full), label=f"I₂/I₁ = {ratio:g},  γ = {gamma:.2f}")
ax1.set_xlabel("разность фаз, градусы")
ax1.set_ylabel("I / (I₁ + I₂)")
ax1.legend(fontsize=8)

r = np.logspace(-3, 0, 200)
ax2.semilogx(r, 2 * np.sqrt(r) / (1 + r))
ax2.set_xlabel("отношение яркостей I₂/I₁")
ax2.set_ylabel("модуляция γ")
plt.show()

for ratio in (1, 0.5, 0.1, 0.04, 0.01):
    print(f"I₂/I₁ = {ratio:<5g}  γ = {2 * np.sqrt(ratio) / (1 + ratio):.3f}")

Модуляция на удивление терпима к неравенству пучков: при отношении яркостей 1 : 10 она ещё 0,57, а при 1 : 25 (стекло без покрытия отражает около 4 %) — 0,38. Полосы видны, даже когда один пучок намного слабее другого.

## 4. Откуда берутся полосы

До сих пор мы смотрели на одну точку. На камере точек много, и разность фаз $\varphi(x, y)$ в каждой своя — потому что разный путь.

Простейший случай: две плоские волны падают на камеру под малым углом $\theta$ друг к другу. Тогда разность хода растёт линейно вдоль $x$:

$$\Delta(x) = x\sin\theta, \qquad \varphi(x) = \frac{2\pi}{\lambda}x\sin\theta$$

Каждый раз, когда $\Delta$ вырастает на $\lambda$, фаза делает полный оборот — получается одна полоса. Период полос:

$$\Lambda = \frac{\lambda}{\sin\theta}$$

In [ ]:
size = 5e-3                                  # размер кадра 5 мм
n = 256
xs = np.linspace(-size / 2, size / 2, n)
X, Y = np.meshgrid(xs, xs)

theta = np.deg2rad(0.05)                     # угол между пучками — три угловые минуты
phi = 2 * np.pi / lam * X * np.sin(theta)
I = 1 + 0.8 * np.cos(phi)

print(f"период полос Λ = {lam / np.sin(theta) * 1e3:.2f} мм,  полос на кадре: {size * np.sin(theta) / lam:.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1.6]})
ax1.imshow(I, cmap="gray", extent=[-2.5, 2.5, -2.5, 2.5], vmin=0, vmax=2)
ax1.set_xlabel("x, мм")
ax1.set_ylabel("y, мм")
ax1.grid(False)
ax1.set_title("интерферограмма")
ax2.plot(xs * 1e3, I[n // 2])
ax2.set_xlabel("x, мм")
ax2.set_ylabel("I")
ax2.set_title("сечение по центральной строке")
plt.show()

Угол в три угловые минуты уже даёт семь полос на пяти миллиметрах. Интерферометр — очень чувствительный прибор для измерения углов и расстояний.

Анимация: постепенно увеличиваем наклон одного пучка. Ноль наклона — равномерно залитое поле («бесконечно широкая полоса»).

In [ ]:
thetas = np.deg2rad(np.linspace(0, 0.08, 40))

fig, ax = plt.subplots(figsize=(4.5, 4.5))
im = ax.imshow(np.zeros((n, n)), cmap="gray", extent=[-2.5, 2.5, -2.5, 2.5], vmin=0, vmax=2)
ax.grid(False)
ax.set_xlabel("x, мм")
ax.set_ylabel("y, мм")
title = ax.set_title("")


def update(i):
    # наклон сразу по двум осям, чтобы полосы были косыми
    phi = 2 * np.pi / lam * (X * np.sin(thetas[i]) + Y * np.sin(thetas[i] / 3))
    im.set_data(1 + 0.8 * np.cos(phi))
    title.set_text(f"наклон = {np.rad2deg(thetas[i]) * 60:.1f} угл. мин")
    return im, title


anim = FuncAnimation(fig, update, frames=len(thetas), interval=120, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

Полосы — это **линии равной разности хода**, как горизонтали на топографической карте. Переход от одной светлой полосы к соседней означает изменение разности хода ровно на $\lambda$. Если один из фронтов не плоский, а с бугром, полосы искривятся — этим займёмся на этапе 3.

## 5. Почему модуляция бывает меньше единицы

Первую причину мы уже видели: неравные яркости пучков. Есть ещё две.

### 5.1. Вибрации за время экспозиции

Если за время, пока открыт затвор, разность фаз успевает подрожать, камера усредняет несколько чуть сдвинутых картин. Светлые полосы наползают на тёмные, контраст падает. Для случайного дрожания фазы с СКО $\sigma$ (в радианах):

$$\gamma \to \gamma\cdot e^{-\sigma^2/2}$$

In [ ]:
sigmas = np.linspace(0, 3, 31)                       # СКО дрожания фазы, рад
gamma_mc = [np.mean(np.cos(rng.normal(0, s, 50_000))) for s in sigmas]

sigma_nm = sigmas * lam / (4 * np.pi) * 1e9          # то же в нанометрах смещения зеркала

fig, ax = plt.subplots()
ax.plot(sigma_nm, gamma_mc, "o", label="усреднение 50 000 случайных сдвигов")
ax.plot(sigma_nm, np.exp(-sigmas**2 / 2), "r", label=r"$e^{-\sigma^2/2}$")
ax.set_xlabel("СКО дрожания зеркала за экспозицию, нм")
ax.set_ylabel("во сколько раз падает модуляция")
ax.legend()
plt.show()

Дрожание зеркала на 50 нм за время кадра уже съедает 40 % контраста, на 100 нм — почти всё. Поэтому экспозицию делают короткой. Пересчёт фазы в нанометры идёт через $\delta = 4\pi\Delta z/\lambda$ — множитель $4\pi$ вместо $2\pi$ из-за того, что свет проходит до зеркала и обратно (этап 4).

Обратите внимание на разницу с разделом 5 этапа 0:

- дрожание **во время** кадра уменьшает модуляцию $b$;
- дрожание **между** кадрами меняет сдвиг $\delta_n$.

Статья борется со вторым.

### 5.2. Когерентность — на уровне идеи

Формула интерференции выведена для одной длины волны. Реальный источник излучает полоску длин волн шириной $\Delta\lambda$. Каждая даёт свою систему полос со своим периодом. При нулевой разности хода все они совпадают, а с ростом разности хода расползаются и замывают друг друга.

In [ ]:
opd = np.linspace(-60e-6, 60e-6, 24_001)             # разность хода, м


def fringes(sigma_lam):
    # Интерференция от источника с гауссовым спектром шириной sigma_lam (СКО).
    if sigma_lam == 0:
        return 1 + np.cos(2 * np.pi * opd / lam)
    lams = np.linspace(lam - 4 * sigma_lam, lam + 4 * sigma_lam, 161)
    w = np.exp(-0.5 * ((lams - lam) / sigma_lam) ** 2)
    w /= w.sum()
    return 1 + (w[:, None] * np.cos(2 * np.pi * opd[None, :] / lams[:, None])).sum(axis=0)


fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, s_nm in zip(axes, (0, 2, 10)):
    ax.plot(opd * 1e6, fringes(s_nm * 1e-9), lw=0.4)
    # огибающая для гауссова спектра: модуляция как функция разности хода
    envelope = np.exp(-0.5 * (2 * np.pi * s_nm * 1e-9 * opd / lam**2) ** 2)
    ax.plot(opd * 1e6, 1 + envelope, "r", lw=1)
    ax.plot(opd * 1e6, 1 - envelope, "r", lw=1)
    label = "идеальный лазер" if s_nm == 0 else f"ширина спектра (СКО) {s_nm} нм"
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("I")
axes[-1].set_xlabel("разность хода, мкм")
plt.tight_layout()
plt.show()

for s_nm in (2, 10):
    half_width = lam**2 / (np.sqrt(2) * np.pi * s_nm * 1e-9)
    print(f"Δλ = {s_nm} нм  →  модуляция падает в e раз при разности хода {half_width * 1e6:.0f} мкм")

Полосы настолько частые, что сливаются в сплошную заливку. Красная огибающая — это модуляция как функция разности хода.

**Длина когерентности** — разность хода, на которой полосы ещё видны. По порядку величины она равна $\lambda^2/\Delta\lambda$; точный множитель зависит от формы спектра и от того, как определить «ещё видны». У светодиода это десятки микрон, у хорошего лазера — метры и больше. Поэтому в интерферометрах для контроля оптики стоит лазер: пути двух пучков могут отличаться на десятки сантиметров, а полосы остаются контрастными.

## Итог

$$I(x,y) = \underbrace{I_1+I_2}_{a(x,y)} + \underbrace{2\sqrt{I_1I_2}}_{b(x,y)}\cos\varphi(x,y)$$

- $a$ и $b$ зависят от координат, потому что пучки освещают поле неравномерно;
- $\varphi(x,y)$ — разность фаз опорной и исследуемой волны; в ней и закодирована форма поверхности;
- камера даёт только $I$. В одном уравнении три неизвестных — отсюда вся дальнейшая история с фазовыми сдвигами.

## Упражнения

1. В разделе 2 сделайте $I_2 = I_1$. Чему равна яркость в максимуме? Почему она вчетверо больше яркости одного пучка, а не вдвое, и не нарушается ли закон сохранения энергии?
2. Какой нужен угол между пучками, чтобы на 5 мм уместилось ровно 20 полос? Проверьте в разделе 4.
3. В разделе 4 замените линейную фазу на квадратичную: `phi = 2*np.pi/lam * (X**2 + Y**2) / (2*R)` с `R = 5` м. Что получилось? Как меняется картина с ростом `R`?
4. При какой СКО дрожания зеркала модуляция падает вдвое? Сравните со смещением, дающим сдвиг фазы 90°.
5. Два пучка от **разных** лазеров полос не дают, хотя формула вроде бы та же. Смоделируйте: пусть разность фаз случайно меняется от 0 до $2\pi$ много раз за экспозицию. Что останется от интерференционного члена?